<a href="https://colab.research.google.com/github/sebastianvettel1212/scikit-learn-cookbook/blob/main/scikit_learn_Cookbook_Chapter2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bab 2 — Alur Pra-Model dan Preprocessing Data (*Pre-Model Workflow and Data Preprocessing*)

**Buku:** *scikit-learn Cookbook, Third Edition* — John Sukup (Packt Publishing)

Notebook ini berisi:
1. **Reproduksi kode** seluruh *recipe* pada Bab 2 (dari repositori resmi buku),
2. **Ringkasan** tiap subbab, dan
3. **Penjelasan teori** dalam Bahasa Indonesia.

---

## Daftar Isi
0. Pengantar: Dampak Data Mentah terhadap Performa Model
1. Menangani Data Hilang (*Missing Data*)
2. Teknik Scaling
3. Encoding Variabel Kategorikal
4. Pengantar Pipeline
5. Visualisasi Pipeline
6. *Feature Engineering*
7. Latihan Praktik: Pipeline Komprehensif
8. Ringkasan Bab

---

## 0. Pengantar: Dampak Data Mentah terhadap Performa Model
Kualitas model sangat bergantung pada kualitas data. Data mentah hampir selalu bermasalah, dan model yang sebaik apa pun akan menghasilkan prediksi buruk bila diberi data buruk (*garbage in, garbage out*).

### Masalah data yang umum
| Masalah | Dampak |
|---|---|
| **Data hilang** (*missing values*) | Banyak algoritma tidak dapat berjalan, atau menghasilkan bias |
| **Skala fitur berbeda** | Model berbasis jarak/gradien (k-NN, SVM, regresi dengan regularisasi) didominasi fitur berskala besar |
| **Variabel kategorikal** | Algoritma membutuhkan angka, bukan teks |
| **Outlier dan noise** | Menggeser parameter model |
| **Fitur tidak informatif/berlebih** | Overfitting dan komputasi lebih berat |

Bab ini menyajikan *recipe* untuk membersihkan dan menyiapkan data: imputasi, scaling, encoding, pipeline, dan *feature engineering*.

## Persiapan

In [1]:
!pip install numpy pandas scikit-learn

In [2]:
import warnings
warnings.filterwarnings('ignore')

---
## 1. Menangani Data Hilang (*Missing Data*)

Banyak algoritma ML tidak menerima nilai kosong (`NaN`). Menghapus baris yang memiliki `NaN` bisa membuang banyak informasi, sehingga umumnya dipakai **imputasi**: mengisi nilai hilang dengan estimasi yang masuk akal.

### Getting ready
Kita membuat dataset mainan berisi 10 fitur numerik acak (20 baris) lalu menyisipkan sekitar 20% nilai hilang secara acak.

In [3]:
# Load libraries
import numpy as np
import pandas as pd

# Create a larger sample dataset with missing values
np.random.seed(2024)  # For reproducibility
n_samples = 20
n_features = 10

# Generate random data
data = {
    f"Feature{i+1}": np.random.uniform(0, 100, n_samples) for i in range(n_features)
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Randomly introduce missing values (approximately 20% of the data)
for column in df.columns:
    mask = np.random.random(n_samples) < 0.2
    df.loc[mask, column] = np.nan

# Display the DataFrame with missing values
display(df)

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,NaN,42.009814,34.680397,49.962259,NaN,NaN,89.954588,41.152421,NaN
1,69.910875,9.554215,6.436369,31.287816,37.966499,NaN,82.005649,NaN,75.797616,91.382492
2,NaN,96.090974,59.643269,84.710402,NaN,84.109027,NaN,70.288128,1.778343,4.117690
3,NaN,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,NaN,NaN,80.077973
4,20.501895,NaN,89.248639,67.655865,58.635861,78.225721,60.911562,NaN,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,NaN,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,NaN,22.966899,21.049022,57.358544,14.302591
7,NaN,NaN,89.538184,69.451294,NaN,47.885551,NaN,33.620401,99.685711,6.683138
8,47.384570,NaN,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,NaN,31.142866,28.865545,NaN,41.004459,41.460336,50.236236,NaN


### How to do it...

### a) `SimpleImputer`
Mengisi nilai hilang dengan **strategi sederhana** per kolom: `"mean"` (rata-rata), `"median"`, `"most_frequent"` (modus), atau `"constant"` (nilai tetap).
- Kelebihan: cepat dan mudah.
- Kekurangan: mengabaikan hubungan antarfitur dan **mengecilkan variansi** data.

In [4]:
# Load libraries
from sklearn.impute import SimpleImputer

# Initialize the SimpleImputer and set the strategy to "mean," "median", or "most_frequent"
imputer = SimpleImputer(strategy="mean")

# Fit and transform the data
imputed_data = imputer.fit_transform(df)
imputed_df = pd.DataFrame(imputed_data, columns=df.columns)
imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,53.864661,42.009814,34.680397,49.962259,57.822964,51.145011,89.954588,41.152421,48.400044
1,69.910875,9.554215,6.436369,31.287816,37.966499,57.822964,82.005649,50.715003,75.797616,91.382492
2,52.558605,96.090974,59.643269,84.710402,46.055883,84.109027,51.145011,70.288128,1.778343,4.117690
3,52.558605,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.715003,60.616723,80.077973
4,20.501895,53.864661,89.248639,67.655865,58.635861,78.225721,60.911562,50.715003,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.055883,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,57.822964,22.966899,21.049022,57.358544,14.302591
7,52.558605,53.864661,89.538184,69.451294,46.055883,47.885551,51.145011,33.620401,99.685711,6.683138
8,47.384570,53.864661,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.897507,31.142866,28.865545,57.822964,41.004459,41.460336,50.236236,48.400044


### b) `KNNImputer`
Mengisi nilai hilang dengan **rata-rata dari tetangga terdekat** (*k-Nearest Neighbors*): baris yang mirip dipakai untuk menaksir nilai yang hilang. Lebih memperhatikan struktur data dibanding `SimpleImputer`, tetapi lebih lambat pada data besar dan sensitif terhadap skala fitur.

In [5]:
# Load libraries
from sklearn.impute import KNNImputer

# Initialize the KNNImputer
knn_imputer = KNNImputer(n_neighbors=2)

# Fit and transform the data using the previously defined DataFrame
knn_imputed_data = knn_imputer.fit_transform(df)
knn_imputed_df = pd.DataFrame(knn_imputed_data, columns=df.columns)
knn_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,48.954043,42.009814,34.680397,49.962259,93.910386,52.549271,89.954588,41.152421,59.833678
1,69.910875,9.554215,6.436369,31.287816,37.966499,86.793468,82.005649,45.416191,75.797616,91.382492
2,67.752344,96.090974,59.643269,84.710402,73.338309,84.109027,59.012876,70.288128,1.778343,4.117690
3,47.880864,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,64.510281,39.726833,80.077973
4,20.501895,31.670912,89.248639,67.655865,58.635861,78.225721,60.911562,25.903612,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,59.716773,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,40.482911,22.966899,21.049022,57.358544,14.302591
7,47.629715,77.843853,89.538184,69.451294,32.753328,47.885551,55.683434,33.620401,99.685711,6.683138
8,47.384570,35.180639,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,29.188271,31.142866,28.865545,59.674651,41.004459,41.460336,50.236236,35.000820


### c) `IterativeImputer`
Memodelkan **tiap fitur yang memiliki nilai hilang sebagai fungsi fitur lain**, lalu mengestimasi nilai hilang secara **iteratif** (mirip pendekatan MICE). Biasanya paling akurat saat antarfitur berkorelasi, tetapi paling mahal secara komputasi. Fitur ini masih *experimental*, sehingga harus diaktifkan lebih dulu.

In [6]:
# Load libraries
from sklearn.experimental import enable_iterative_imputer # Experimental feature requires loading
from sklearn.impute import IterativeImputer

# Initialize the IterativeImputer
iterative_imputer = IterativeImputer()

# Fit and transform the data using the previously defined DataFrame
iterative_imputed_data = iterative_imputer.fit_transform(df)
iterative_imputed_df = pd.DataFrame(iterative_imputed_data, columns=df.columns)
iterative_imputed_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,58.801452,54.365008,42.009814,34.680397,49.962259,58.680582,51.178102,89.954588,41.152421,48.292275
1,69.910875,9.554215,6.436369,31.287816,37.966499,60.070634,82.005649,50.710323,75.797616,91.382492
2,52.539676,96.090974,59.643269,84.710402,46.121290,84.109027,51.155371,70.288128,1.778343,4.117690
3,52.630591,25.176729,83.732372,88.023110,16.886931,97.205554,84.696823,50.680265,50.650877,80.077973
4,20.501895,53.198954,89.248639,67.655865,58.635861,78.225721,60.911562,50.691674,65.114243,99.119187
5,10.606287,76.825393,20.052744,5.367515,46.156310,19.703051,34.423301,93.399494,72.206680,12.640276
6,72.724014,79.792340,50.239523,55.921377,6.191019,55.923578,22.966899,21.049022,57.358544,14.302591
7,52.558131,53.942885,89.538184,69.451294,46.070477,47.885551,50.969808,33.620401,99.685711,6.683138
8,47.384570,54.204978,25.592093,82.419730,73.414540,61.663700,29.172571,65.946718,61.005155,34.052747
9,44.829582,38.165095,52.882614,31.142866,28.865545,55.978300,41.004459,41.460336,50.236236,48.429901


### There's more... (membandingkan ketiga imputer)
Kita bisa memeriksa apakah masih ada nilai hilang, dan membandingkan hasil imputasi pada satu sel yang sama.

In [7]:
# Tambahan (tidak ada di kode buku): bandingkan hasil imputasi pada sel yang awalnya hilang
mask_na = df.isna()
r, c = np.argwhere(mask_na.values)[0]
print(f'Sel yang hilang: baris {r}, kolom {df.columns[c]}')
print('SimpleImputer   :', round(imputed_df.iloc[r, c], 3))
print('KNNImputer      :', round(knn_imputed_df.iloc[r, c], 3))
print('IterativeImputer:', round(iterative_imputed_df.iloc[r, c], 3))
print('Sisa NaN setelah imputasi:', int(imputed_df.isna().sum().sum()))

Sel yang hilang: baris 0, kolom Feature2
SimpleImputer   : 53.865
KNNImputer      : 48.954
IterativeImputer: 54.365
Sisa NaN setelah imputasi: 0


### Key Ideas
- Imputasi mengisi nilai hilang agar data bisa dipakai model.
- `SimpleImputer` (cepat, sederhana) → `KNNImputer` (berbasis kemiripan) → `IterativeImputer` (berbasis model antarfitur).
- Lakukan `fit` hanya pada data latih untuk menghindari *data leakage*.

---
## 2. Teknik Scaling

Scaling dan normalisasi memastikan **setiap fitur berkontribusi seimbang**, terutama pada algoritma yang bergantung pada jarak (k-NN, SVM) atau gradien. Fitur berskala besar (mis. gaji) akan mendominasi fitur berskala kecil (mis. usia) bila tidak diskalakan.

### Getting ready
Memakai `iterative_imputed_df` dari recipe sebelumnya.

### a) `StandardScaler`
Menghilangkan rata-rata dan membagi dengan simpangan baku (z-score): $z=(x-\mu)/\sigma$. Hasil: rata-rata 0 dan simpangan baku 1. Cocok bila data kira-kira normal; sensitif terhadap outlier.

In [8]:
# Load libraries
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Fit and transform the data using the iterative imputed DataFrame
scaled_data = scaler.fit_transform(iterative_imputed_df)
scaled_df = pd.DataFrame(scaled_data, columns=iterative_imputed_df.columns)
scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.271931,0.019495,-0.373057,-0.894258,0.188761,0.031844,0.001905,1.683800,-0.780898,-0.003574
1,0.756048,-1.874939,-1.592196,-1.043864,-0.391744,0.085949,1.426871,-0.000534,0.652484,1.429056
2,-0.000939,1.783513,0.231259,1.311954,0.002887,1.021588,0.000854,0.839730,-2.409929,-1.472256
3,0.003022,-1.214477,1.056818,1.458037,-1.411839,1.531340,1.551267,-0.001824,-0.387917,1.053212
4,-1.397054,-0.029802,1.245866,0.559887,0.608499,0.792594,0.451822,-0.001334,0.210478,1.686279
5,-1.828276,0.969036,-1.125549,-2.186891,0.004582,-1.485267,-0.772566,1.831653,0.503915,-1.188904
6,0.878636,1.094467,-0.091017,0.042422,-1.929442,-0.075466,-1.302124,-1.273574,-0.110399,-1.133636
7,-0.000135,0.001649,1.255789,0.639061,0.000428,-0.388327,-0.007723,-0.734020,1.640810,-1.386962
8,-0.225584,0.012729,-0.935710,1.210940,1.323678,0.147955,-1.015275,0.653400,0.040472,-0.476999
9,-0.336923,-0.665377,-0.000435,-1.050256,-0.832163,-0.073336,-0.468360,-0.397536,-0.405072,0.001002


### b) `MinMaxScaler`
Menskalakan tiap fitur ke rentang tertentu (default **0 sampai 1**): $x'=\dfrac{x-x_{min}}{x_{max}-x_{min}}$. Bentuk distribusi tidak berubah, tetapi sangat sensitif terhadap outlier.

In [9]:
# Load libraries
from sklearn.preprocessing import MinMaxScaler

# Initialize the MinMaxScaler
minmax_scaler = MinMaxScaler()

# Fit and transform the data using the iterative imputed DataFrame
minmax_scaled_data = minmax_scaler.fit_transform(iterative_imputed_df)
minmax_scaled_df = pd.DataFrame(
    minmax_scaled_data, columns=iterative_imputed_df.columns
)
minmax_scaled_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.603506,0.517824,0.380540,0.354639,0.569020,0.555896,0.435006,0.962767,0.402156,0.464988
1,0.721357,0.000000,0.000369,0.313594,0.413077,0.571920,0.833074,0.538604,0.756013,0.918562
2,0.537080,1.000000,0.568987,0.959922,0.519088,0.849027,0.434713,0.750206,0.000000,0.000000
3,0.538045,0.180530,0.826426,1.000000,0.139045,1.000000,0.867825,0.538279,0.499171,0.799569
4,0.197218,0.504349,0.885377,0.753589,0.681776,0.781206,0.560692,0.538402,0.646896,1.000000
5,0.092244,0.777371,0.145886,0.000000,0.519543,0.106575,0.218656,1.000000,0.719336,0.089710
6,0.751199,0.811657,0.468490,0.611621,0.000000,0.524114,0.070723,0.218016,0.567681,0.107208
7,0.537276,0.512946,0.888472,0.775311,0.518428,0.431454,0.432317,0.353891,1.000000,0.027004
8,0.482394,0.515975,0.205085,0.932208,0.873897,0.590284,0.150855,0.703283,0.604927,0.315101
9,0.455290,0.330621,0.496737,0.311840,0.294766,0.524745,0.303637,0.438627,0.494936,0.466437


### c) `Normalizer`
Bekerja pada **tiap baris (sampel)**, bukan kolom: menskalakan setiap sampel agar memiliki **norma satuan** (panjang vektor = 1). Berguna pada data teks atau saat arah vektor lebih penting daripada besarnya.

In [10]:
# Load libraries
from sklearn.preprocessing import Normalizer

# Initialize the Normalizer
normalizer = Normalizer()

# Fit and transform the data using the iterative imputed DataFrame
normalized_data = normalizer.fit_transform(iterative_imputed_df)
normalized_df = pd.DataFrame(normalized_data, columns=iterative_imputed_df.columns)
normalized_df

,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10
0,0.339168,0.313579,0.242313,0.200037,0.288183,0.338471,0.295196,0.518860,0.237368,0.278551
1,0.376706,0.051482,0.034682,0.168591,0.204578,0.323683,0.441878,0.273247,0.408426,0.492404
2,0.264336,0.483450,0.300075,0.426192,0.232044,0.423166,0.257371,0.353631,0.008947,0.020717
3,0.243762,0.116607,0.387811,0.407684,0.078213,0.450213,0.392278,0.234729,0.234592,0.370886
4,0.095909,0.248868,0.417511,0.316499,0.274302,0.365945,0.284948,0.237139,0.304609,0.463686
5,0.068115,0.493382,0.128781,0.034471,0.296421,0.126535,0.221071,0.599823,0.463720,0.081177
6,0.460521,0.505281,0.318139,0.354119,0.039204,0.354133,0.145437,0.133292,0.363220,0.090570
7,0.274582,0.281816,0.467778,0.362837,0.240688,0.250170,0.266284,0.175644,0.520792,0.034915
8,0.265283,0.303467,0.143277,0.461427,0.411012,0.345225,0.163323,0.369203,0.341538,0.190645
9,0.321287,0.273524,0.379002,0.223196,0.206875,0.401188,0.293873,0.297140,0.360036,0.347090


In [11]:
# Tambahan (tidak ada di kode buku): verifikasi sifat tiap scaler
print('StandardScaler -> mean ~0 :', np.round(scaled_df.mean().mean(), 6),
      '| std ~1 :', np.round(scaled_df.std(ddof=0).mean(), 6))
print('MinMaxScaler   -> min/max :', minmax_scaled_df.min().min(), '/', minmax_scaled_df.max().max())
print('Normalizer     -> norma tiap baris:', np.round(np.linalg.norm(normalized_df, axis=1)[:5], 3))

StandardScaler -> mean ~0 : -0.0 | std ~1 : 1.0
MinMaxScaler   -> min/max : 0.0 / 1.0000000000000002
Normalizer     -> norma tiap baris: [1. 1. 1. 1. 1.]


**Interpretasi:** `StandardScaler` menghasilkan rata-rata ≈ 0 dan std ≈ 1; `MinMaxScaler` memetakan ke [0, 1]; `Normalizer` membuat norma tiap **baris** = 1.

### Key Ideas
- `StandardScaler` (z-score), `MinMaxScaler` (rentang 0–1), `Normalizer` (norma satuan per sampel).
- Scaling wajib untuk model berbasis jarak/gradien; tidak diperlukan untuk model berbasis pohon.

---
## 3. Encoding Variabel Kategorikal

Algoritma ML membutuhkan input numerik, sehingga variabel kategorikal (teks) harus **di-encode**.

### Getting ready
Dataset mainan dengan tiga variabel kategorikal: `Department`, `Position`, dan `Location` (20 record).

In [12]:
# Load libraries
import numpy as np

# Create sample categorical data with 20 records
np.random.seed(2024)  # for reproducibility
categories = ["A", "B", "C", "D"]
categorical_data = pd.DataFrame(
    {
        "Department": np.random.choice(categories, size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
        "Location": np.random.choice(["NY", "SF", "LA", "CHI"], size=20),
    }
)

# Display the DataFrame with categorical values
display(categorical_data)

,Department,Position,Location
0,A,Manager,LA
1,C,Manager,LA
2,A,Junior,NY
3,A,Manager,LA
4,D,Manager,NY
5,A,Senior,LA
6,C,Manager,SF
7,D,Manager,LA
8,B,Junior,LA
9,D,Manager,CHI


### a) `OneHotEncoder`
Mengubah tiap kategori menjadi **kolom biner (0/1)** terpisah. Cocok untuk variabel **nominal** (tanpa urutan), karena tidak menciptakan urutan palsu antar kategori. Kekurangannya: jumlah kolom bertambah banyak bila kategori banyak (*high cardinality*).

In [13]:
# Load libraries
from sklearn.preprocessing import OneHotEncoder

# Initialize the OneHotEncoder
onehot_encoder = OneHotEncoder(sparse_output=False)

# Fit and transform the data
onehot_encoded_data = onehot_encoder.fit_transform(categorical_data)
onehot_encoded_df = pd.DataFrame(
    onehot_encoded_data, columns=onehot_encoder.get_feature_names_out()
)
onehot_encoded_df

,Department_A,Department_B,Department_C,Department_D,Position_Junior,Position_Manager,Position_Senior,Location_CHI,Location_LA,Location_NY,Location_SF
0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
5,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
7,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
8,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0


### b) `LabelEncoder`
Memberi kode bilangan bulat `0` sampai `n_classes-1` pada tiap kategori. Dirancang untuk **label target (`y`)**, bukan fitur (`X`). Memakainya pada fitur nominal menciptakan urutan semu (mis. "NY" < "SF"). Untuk fitur ordinal, gunakan `OrdinalEncoder`.

In [14]:
# Load libraries
from sklearn.preprocessing import LabelEncoder

# Initialize the LabelEncoder
label_encoder = LabelEncoder()

# Create a new DataFrame to store label encoded values
label_encoded_df = pd.DataFrame()

# Fit and transform each categorical column
for column in categorical_data.columns:
    label_encoded_df[f"{column}_encoded"] = label_encoder.fit_transform(
        categorical_data[column]
    )
label_encoded_df

,Department_encoded,Position_encoded,Location_encoded
0,0,1,1
1,2,1,1
2,0,0,2
3,0,1,1
4,3,1,2
5,0,2,1
6,2,1,3
7,3,1,1
8,1,0,1
9,3,1,0


### c) `ColumnTransformer`
Memungkinkan **transformasi berbeda untuk kolom berbeda**, lalu menggabungkan hasilnya menjadi satu ruang fitur. Contoh: kolom numerik di-scale, kolom kategorikal di-*one-hot*. Parameter `remainder="passthrough"` membiarkan kolom lain tetap apa adanya.

In [15]:
# Load libraries
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Create sample mixed data with 20 records
np.random.seed(2024)  # for reproducibility
mixed_data = pd.DataFrame(
    {
        "Age": np.random.randint(25, 65, size=20),
        "Salary": np.round(np.random.normal(60000, 15000, size=20), 2),
        "Experience": np.random.randint(1, 20, size=20),
        "Department": np.random.choice(["IT", "HR", "Sales", "Finance"], size=20),
        "Position": np.random.choice(["Junior", "Senior", "Manager"], size=20),
    }
)

# Display the DataFrame with mixed data
display(mixed_data)

# Initialize the ColumnTransformer
column_transformer = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary", "Experience"]),
        ("cat", OneHotEncoder(), ["Department", "Position"]),
    ],
    remainder="passthrough",
)

# Fit and transform the data
transformed_data = column_transformer.fit_transform(mixed_data)

# Get feature names for the transformed columns
numeric_cols = ["Age_scaled", "Salary_scaled", "Experience_scaled"]
categorical_cols = column_transformer.named_transformers_["cat"].get_feature_names_out(
    ["Department", "Position"]
)

# Create the transformed DataFrame
transformed_df = pd.DataFrame(
    transformed_data, columns=numeric_cols + list(categorical_cols)
)
transformed_df

,Age,Salary,Experience,Department,Position
0,33,59420.36,12,Finance,Manager
1,57,82895.92,17,Sales,Junior
2,25,38165.76,16,Finance,Manager
3,52,38242.36,7,IT,Junior
4,61,55088.65,8,Sales,Manager
5,26,78688.88,9,Sales,Senior
6,60,49585.62,14,Finance,Junior
7,35,47992.58,17,HR,Manager
8,27,54833.02,9,Sales,Senior
9,57,93535.56,14,Sales,Senior


,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,Position_Senior
0,-1.045349,-0.305043,0.327303,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.727681,1.105091,1.262454,0.0,0.0,0.0,1.0,1.0,0.0,0.0
2,-1.636358,-1.581768,1.075424,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.358300,-1.577167,-0.607848,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,1.023186,-0.565241,-0.420818,0.0,0.0,0.0,1.0,0.0,1.0,0.0
5,-1.562482,0.852382,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
6,0.949309,-0.895798,0.701364,1.0,0.0,0.0,0.0,1.0,0.0,0.0
7,-0.897596,-0.991489,1.262454,0.0,1.0,0.0,0.0,0.0,1.0,0.0
8,-1.488606,-0.580596,-0.233788,0.0,0.0,0.0,1.0,0.0,0.0,1.0
9,0.727681,1.744195,0.701364,0.0,0.0,0.0,1.0,0.0,0.0,1.0


### Key Ideas
- `OneHotEncoder` untuk kategori nominal; `LabelEncoder` untuk label target; `OrdinalEncoder` untuk kategori berurutan.
- `ColumnTransformer` menerapkan transformasi berbeda pada kolom berbeda dalam satu langkah.

---
## 4. Pengantar Pipeline

**Pipeline** menggabungkan rangkaian transformer dan (opsional) satu estimator akhir ke dalam satu objek. Manfaat: kode lebih rapi, langkah selalu berurutan, **mencegah data leakage**, dan seluruh alur dapat disetel/divalidasi bersama.

### Getting ready
Sintaks umum:
```python
pipeline = Pipeline([
    ("nama langkah", transformer),
    ("nama langkah", transformer),
    ...,
    ("nama langkah", estimator),
])
```
Semua langkah kecuali yang terakhir harus **transformer** (punya `fit`/`transform`); langkah terakhir bisa transformer atau estimator.

### How to do it...
Pipeline dasar: imputasi nilai hilang → scaling. Pertama, kita pisahkan fitur dan target (kolom terakhir dianggap target) dan bagi data menjadi latih/uji.

In [16]:
# Load libraries
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# First, separate features and target (assuming last column is target)
X = transformed_df.iloc[:, :-1]  # all columns except last
y = transformed_df.iloc[:, -1]  # last column

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Create a pipeline
pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="mean")),  # handle missing values
        ("scaler", StandardScaler()),  # scale the features
    ]
)

# Fit and transform the training data
X_train_transformed = pipeline.fit_transform(X_train)

# Transform the test data
X_test_transformed = pipeline.transform(X_test)

# Create DataFrames with the transformed data (to preserve column names)
X_train_transformed = pd.DataFrame(
    X_train_transformed, columns=X_train.columns, index=X_train.index
)

X_test_transformed = pd.DataFrame(
    X_test_transformed, columns=X_test.columns, index=X_test.index
)

In [17]:
# Tambahan (tidak ada di kode buku): hasil transformasi data latih
X_train_transformed.head()

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
13,0.834298,-0.867451,-1.559779,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,-1.0
12,-0.689202,1.282818,-0.150946,-0.57735,2.081666,-0.480384,-0.774597,-0.57735,1.0
16,0.544107,-1.173357,-1.761041,-0.57735,-0.480384,2.081666,-0.774597,-0.57735,1.0
5,-1.342131,0.821525,-0.150946,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,-1.0
17,1.414679,0.611115,0.654101,-0.57735,-0.480384,-0.480384,1.290994,-0.57735,1.0


**Interpretasi:** `fit_transform()` dipakai pada data latih, sedangkan data uji hanya `transform()` dengan parameter dari data latih (lihat Bab 1).

---
## 5. Visualisasi Pipeline

Visualisasi membantu memeriksa urutan dan konfigurasi tiap langkah. Setelah `set_config(display="diagram")`, objek pipeline ditampilkan sebagai diagram interaktif di notebook.

In [18]:
# Load libraries
from sklearn import set_config

# Set display configuration
set_config(display="diagram")

# Display the pipeline
pipeline

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler())])

### Key Ideas
- Pipeline = rangkaian langkah preprocessing (+ model) dalam satu objek.
- `fit_transform` pada data latih, `transform` pada data uji; pipeline melakukannya otomatis dan konsisten.
- `set_config(display="diagram")` menampilkan struktur pipeline.

---
## 6. *Feature Engineering*

**Feature engineering** adalah membuat atau memodifikasi fitur agar model lebih mudah menangkap pola, sering kali lebih berpengaruh daripada pemilihan algoritma. Pada bagian ini dipakai `X_train_transformed` dari recipe sebelumnya.

### a) `PolynomialFeatures`
Menghasilkan fitur **polinomial dan interaksi**, mis. untuk fitur $a$ dan $b$ derajat 2: $1, a, b, a^2, ab, b^2$. Memungkinkan model linear menangkap hubungan **non-linear**, dengan risiko jumlah fitur melonjak.

In [19]:
# Load libraries
from sklearn.preprocessing import PolynomialFeatures

# Initialize the PolynomialFeatures
poly = PolynomialFeatures(degree=2)

# Fit and transform the X_train_transformed data
poly_features = poly.fit_transform(X_train_transformed)
poly_features_df = pd.DataFrame(
    poly_features, columns=poly.get_feature_names_out(X_train_transformed.columns)
)
poly_features_df

,1,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager,...,Department_IT^2,Department_IT Department_Sales,Department_IT Position_Junior,Department_IT Position_Manager,Department_Sales^2,Department_Sales Position_Junior,Department_Sales Position_Manager,Position_Junior^2,Position_Junior Position_Manager,Position_Manager^2
0,1.0,0.834298,-0.867451,-1.559779,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,-1.0,...,4.333333,-1.612452,-1.201850,-2.081666,0.600000,0.447214,0.774597,0.333333,0.577350,1.0
1,1.0,-0.689202,1.282818,-0.150946,-0.577350,2.081666,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
2,1.0,0.544107,-1.173357,-1.761041,-0.577350,-0.480384,2.081666,-0.774597,-0.577350,1.0,...,4.333333,-1.612452,-1.201850,2.081666,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
3,1.0,-1.342131,0.821525,-0.150946,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0
4,1.0,1.414679,0.611115,0.654101,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,1.0,...,0.230769,-0.620174,0.277350,-0.480384,1.666667,-0.745356,1.290994,0.333333,-0.577350,1.0
5,1.0,-1.414679,-1.479801,1.257887,1.732051,-0.480384,-0.480384,-0.774597,-0.577350,1.0,...,0.230769,0.372104,0.277350,-0.480384,0.600000,0.447214,-0.774597,0.333333,-0.577350,1.0
6,1.0,1.269583,0.576819,0.452839,1.732051,-0.480384,-0.480384,-0.774597,1.732051,-1.0,...,0.230769,0.372104,-0.832050,0.480384,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
7,1.0,0.544107,-1.475450,-0.553470,-0.577350,-0.480384,2.081666,-0.774597,1.732051,-1.0,...,4.333333,-1.612452,3.605551,-2.081666,0.600000,-1.341641,0.774597,3.000000,-1.732051,1.0
8,1.0,0.906845,1.060444,1.459148,-0.577350,-0.480384,-0.480384,1.290994,1.732051,-1.0,...,0.230769,-0.620174,-0.832050,0.480384,1.666667,2.236068,-1.290994,3.000000,-1.732051,1.0
9,1.0,0.906845,1.664674,0.855363,-0.577350,-0.480384,-0.480384,1.290994,-0.577350,-1.0,...,0.230769,-0.620174,0.277350,0.480384,1.666667,-0.745356,-1.290994,0.333333,0.577350,1.0


### b) `KBinsDiscretizer`
**Diskretisasi** fitur kontinu menjadi $k$ *bin*. Parameter `strategy`: `"uniform"` (lebar sama), `"quantile"` (jumlah data sama), `"kmeans"`. Berguna untuk menangkap efek non-linear atau mengurangi pengaruh outlier.

In [20]:
# Load libraries
from sklearn.preprocessing import KBinsDiscretizer

# Initialize the KBinsDiscretizer
kbins = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="uniform")

# Fit and transform the X_train_transformed data
binned_data = kbins.fit_transform(X_train_transformed)
binned_df = pd.DataFrame(binned_data, columns=X_train_transformed.columns)
binned_df

,Age_scaled,Salary_scaled,Experience_scaled,Department_Finance,Department_HR,Department_IT,Department_Sales,Position_Junior,Position_Manager
0,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0
1,0.0,2.0,1.0,0.0,2.0,0.0,0.0,0.0,2.0
2,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,2.0
3,0.0,2.0,1.0,0.0,0.0,0.0,2.0,0.0,0.0
4,2.0,1.0,2.0,0.0,0.0,0.0,2.0,0.0,2.0
5,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,2.0
6,2.0,1.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0
7,2.0,0.0,1.0,0.0,0.0,2.0,0.0,2.0,0.0
8,2.0,2.0,2.0,0.0,0.0,0.0,2.0,2.0,0.0
9,2.0,2.0,2.0,0.0,0.0,0.0,2.0,0.0,0.0


### c) `RFE` (*Recursive Feature Elimination*)
Memilih fitur dengan **menghapus fitur paling tidak penting secara berulang** berdasarkan peringkat penting dari estimator. `ranking_` = 1 berarti fitur terbaik; makin besar angkanya, makin awal fitur itu dibuang.

In [21]:
# Load libraries
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

# Initialize the RFE
rfe = RFE(estimator=LinearRegression(), n_features_to_select=1)

# Fit the RFE to the X_train_transformed and y_train data
rfe.fit(X_train_transformed, y_train)

# Get the ranking of features
rfe.ranking_

array([5, 3, 8, 7, 6, 4, 9, 2, 1])

### d) `SelectFromModel`
Memilih fitur berdasarkan **bobot kepentingan** dari model yang sudah dilatih (koefisien atau `feature_importances_`). Fitur dengan kepentingan di bawah `threshold` (di sini rata-rata) dibuang.

In [22]:
# Load libraries
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LinearRegression

# Initialize SelectFromModel with LinearRegression
selector = SelectFromModel(
    estimator=LinearRegression(),
    prefit=False,
    threshold="mean",  # Use mean of feature importances as threshold
)

# Fit the selector
selector.fit(X_train_transformed, y_train)

# Get selected features
selected_features_mask = selector.get_support()

# Get feature names that were selected
selected_features = X_train_transformed.columns[selected_features_mask].tolist()

# Print feature importance scores and selection status
feature_importance = pd.DataFrame(
    {
        "Feature": X_train_transformed.columns,
        "Importance": selector.estimator_.coef_,
        "Selected": selected_features_mask,
    }
)
feature_importance.sort_values("Importance", key=abs, ascending=False)

,Feature,Importance,Selected
8,Position_Manager,-5.000000e-01,True
7,Position_Junior,-4.330127e-01,True
2,Experience_scaled,-1.191755e-15,False
1,Salary_scaled,9.298118e-16,False
0,Age_scaled,6.755007e-16,False
3,Department_Finance,5.342948e-16,False
4,Department_HR,-2.983724e-16,False
5,Department_IT,-2.810252e-16,False
6,Department_Sales,-1.561251e-16,False


> **Catatan:** data pada recipe ini sangat kecil (≈16 baris) dan hanya berupa contoh acak, sehingga peringkat/seleksi fitur di atas bukan temuan bermakna, melainkan sekadar demonstrasi cara memakai API.

### Key Ideas
- `PolynomialFeatures` dan `KBinsDiscretizer` **membuat/mengubah** fitur.
- `RFE` dan `SelectFromModel` **memilih** fitur yang paling berguna.
- Feature engineering yang baik sering mengalahkan penyetelan algoritma.

---
## 7. Latihan Praktik: Pipeline Komprehensif

Menggabungkan semua recipe: **imputasi → scaling → model** pada dataset **California Housing** (`RandomForestRegressor`). Skor yang dilaporkan `score()` untuk regresor adalah $R^2$ pada data uji.

> **Catatan:** `fetch_california_housing()` mengunduh dataset dari internet saat pertama kali dijalankan, jadi pastikan koneksi internet aktif (Google Colab sudah mendukung).

In [23]:
# Load libraries
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

# Load the California Housing dataset
california_data = fetch_california_housing()
X, y = california_data.data, california_data.target

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2024
)

# Create a comprehensive pipeline
comprehensive_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", RandomForestRegressor()),
    ]
)

# Fit the pipeline
comprehensive_pipeline.fit(X_train, y_train)

# Evaluate the pipeline
score = comprehensive_pipeline.score(X_test, y_test)
score

0.8026768047828426

**Interpretasi:** nilai $R^2$ yang dihasilkan menunjukkan seberapa besar variasi harga rumah di data uji yang dapat dijelaskan model (semakin mendekati 1 semakin baik). Karena `RandomForestRegressor` tidak diberi `random_state`, angkanya dapat sedikit berbeda tiap dijalankan.

---
## 8. Ringkasan Bab

Bab ini menunjukkan bahwa **kualitas model dimulai dari kualitas data**. Kita belajar mengatasi data hilang (imputasi), menyamakan skala (scaling), mengubah kategori menjadi angka (encoding), merangkai semuanya dalam pipeline, dan memperkaya/memilih fitur melalui *feature engineering*.

### Rangkuman Konsep dan Fungsi
| Subbab | Konsep utama | Kelas/Fungsi scikit-learn |
|---|---|---|
| Data hilang | Imputasi sederhana, berbasis tetangga, iteratif | `SimpleImputer`, `KNNImputer`, `IterativeImputer` |
| Scaling | Z-score, rentang 0–1, norma satuan | `StandardScaler`, `MinMaxScaler`, `Normalizer` |
| Encoding | Kategori → angka | `OneHotEncoder`, `LabelEncoder`, `ColumnTransformer` |
| Pipeline | Rangkaian langkah berurutan | `Pipeline`, `set_config(display="diagram")` |
| Feature engineering | Membuat & memilih fitur | `PolynomialFeatures`, `KBinsDiscretizer`, `RFE`, `SelectFromModel` |
| Latihan | Pipeline end-to-end | `fetch_california_housing`, `RandomForestRegressor` |

**Pesan utama:** bangun preprocessing sebagai **pipeline** agar konsisten, bebas *data leakage*, dan mudah direproduksi.